# 03 — Steering Criteria Evaluator

**The value.** Sometimes you don't have ground truth and you don't want to design a metric — you just want the prompt to produce outputs in a certain *shape*: one declarative sentence, JSON only, no apologies, always include LaTeX. Steering lets you state those rules in plain English and have APO's internal scorer enforce them. Zero infrastructure (no Lambda, no judge model), lowest setup friction of the three modes.

**Why this method.** Pick steering when the goal is format, structure, or tone shaping rather than correctness against a reference. It's the fastest path from "I want the model to behave like X" to a tuned prompt — usually a few minutes of writing 3–5 rules. Trade-off: you cap at 5 criteria and don't get per-sample scoring telemetry, so it's not a substitute for a real metric when correctness matters.

**How it works.** You pass a `steeringCriteria: [str, ...]` list of plain-English rules (≤5). For each candidate × sample, APO's built-in scorer reads the model output, checks adherence to each rule, and folds the results into a single score per sample. `referenceResponse` is still passed and used as soft guidance, but it's not the optimization target.

**You will learn**
- How `steeringCriteria` slots into the input record (≤ 5 strings).
- Why steering shines for format/structure/tone shaping when there's no clean numeric metric.
- That sample-level `referenceResponse` is still passed (used as soft guidance).

**Two Scenarios**
1. **Text** — XSum (single-sentence news summarization with 5 style rules).
2. **Image** — MathVista (visual math reasoning with an answer-first + LaTeX format).

> **Tip:** set `APO_USE_REFERENCE=1` to replay both chapters from bundled results.

In [ ]:
import utils
from IPython.display import display, Markdown
utils.render_mode_overview("steering")

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown

import utils

env = utils.load_env()
bedrock, s3, lambda_client = utils.make_clients(env)

# Default target model matches the bundle reference run. Edit and re-execute to try other models.
TARGET_MODEL_ID = "us.anthropic.claude-opus-4-6-v1"


# Replay-mode toggle. Set `APO_USE_REFERENCE=1` before launching Jupyter to skip live AWS calls.
USING_REPLAY = os.environ.get("APO_USE_REFERENCE") == "1"
display(Markdown(f"**Live mode:** `{not USING_REPLAY}` &nbsp;|&nbsp; **Bucket:** `{env['BUCKET']}` &nbsp;|&nbsp; **Region:** `{env['REGION']}`"))

---
## Chapter A — Text (XSum)

XSum is a news summarization dataset. Each sample has a multi-paragraph article and a one-sentence reference summary. The five steering rules below tell the optimizer what a *good* summary looks like — concise, factual, single-sentence, 5W frame.

In [ ]:
XSUM_CRITERIA = ['Produce exactly one declarative sentence summarizing the article.', 'The summary must be faithful to the source — do not introduce facts not present in the article.', 'The summary must be at most 25 words.', 'The summary must avoid opinionated, evaluative, or speculative language.', 'The single sentence should capture the WHO, WHAT, and WHEN of the article (5W frame).']
for i, c in enumerate(XSUM_CRITERIA, 1):
    print(f"{i}. {c}")

In [ ]:
preview = utils.build_steering_record("xsum", steering_criteria=XSUM_CRITERIA, bucket=env["BUCKET"])
display(utils.render_dimensions(preview))
display(Markdown("**First sample shape:**"))
display(utils.render_sample_shape(preview))

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
record = utils.build_steering_record("xsum", steering_criteria=XSUM_CRITERIA, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

xsum_results = utils.run_job(
    record,
    example="xsum",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {xsum_results}")

In [ ]:
parsed = utils.parse_results(xsum_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

**A note on polling.** `utils.run_job(...)` orchestrates upload → submit → poll → download. We pass it an `on_status` callback that overwrites a single Markdown line in place — no scrolling log spam over the 20–50 minute job run.

If `APO_USE_REFERENCE=1` is set, `run_job` returns the bundled `reference_results.jsonl` without calling AWS.

---
## Chapter B — Image (MathVista)

MathVista shows the model a chart, table, or diagram and asks a math/reasoning question. We steer toward an **answer-first** response format with **LaTeX-rendered derivations** that explicitly reference the visual elements.

This is the canonical case where steering wins: there's no easy numeric metric for "good explanation format", but the criteria are easy to state.

In [ ]:
# Swap target to a vision-reasoning model.
TARGET_MODEL_ID = "qwen.qwen3-vl-235b-a22b"

if not USING_REPLAY:
    utils.sync_assets("mathvista", env, s3=s3)

In [ ]:
MATHVISTA_CRITERIA = ['State the final answer first on its own line, in the format `Answer: <X>` (a number, an option letter, or short text).', 'After the answer line, provide step-by-step mathematical derivations grounded in the image, with each derivation step rendered in LaTeX (`$...$` for inline math, `$$...$$` for display equations).', 'Reasoning must reference specific visual elements of the image (axes, labels, shapes, colors, table cells, etc.) where relevant.', 'Do not output any prose outside of the answer line and the LaTeX-formatted derivation steps.']
for i, c in enumerate(MATHVISTA_CRITERIA, 1):
    print(f"{i}. {c}")

In [ ]:
preview = utils.build_steering_record("mathvista", steering_criteria=MATHVISTA_CRITERIA, bucket=env["BUCKET"])
display(utils.render_dimensions(preview))
display(Markdown("**First sample shape (image attached as `inputVariablesMultimodal`):**"))
display(utils.render_sample_shape(preview))

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
record = utils.build_steering_record("mathvista", steering_criteria=MATHVISTA_CRITERIA, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

mathvista_results = utils.run_job(
    record,
    example="mathvista",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {mathvista_results}")

In [ ]:
parsed = utils.parse_results(mathvista_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

---
## Recap

You ran two steering jobs without writing a single line of metric code — just a list of rules per record. That's all three APO evaluator modes covered. Pick the one that fits your task:

| You have… | Use… |
|---|---|
| A programmable, deterministic metric | Lambda |
| A subjective quality judgement (with or without a reference) | LLM-as-Judge |
| Format / structure / tone rules and no numeric metric | Steering criteria |